<a href="https://colab.research.google.com/github/Saathvik-Choudhary/Data_Profiling_Chatbot/blob/jupyter-notebook/Data_Profiling_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers accelerate torch gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.

In [ ]:
import sqlite3
import re
import logging
from datetime import datetime, timedelta
from typing import Optional, List, Dict, Any, Tuple
from transformers import pipeline
from contextlib import contextmanager

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [ ]:
print("Loading Phi-3-mini-4k-instruct model...")
pipe = pipeline(
    "text-generation",
    model="microsoft/Phi-3-mini-4k-instruct",
    device_map="auto"
)
print("Model loaded successfully!")

Loading Phi-3-mini-4k-instruct model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Device set to use cuda:0


Model loaded successfully!


In [ ]:
class Settings:
    """Application settings."""
    # Database settings
    db_database: str = "profiling_sample.db"

    # Allowed database views (security constraint)
    allowed_views: list = [
        "profiling_column_stats",
        "profiling_table_stats",
        "profiling_data_quality"
    ]

    # Log level
    log_level: str = "INFO"

settings = Settings()

In [ ]:
class DatabaseConnection:
    """Manages SQLite database connections and query execution."""

    def __init__(self, db_path: str = "profiling_sample.db"):
        self.db_path = db_path

    @contextmanager
    def get_connection(self):
        """Context manager for database connections."""
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row  # Enable dict-like access
        try:
            logger.info(f"SQLite connection established: {self.db_path}")
            yield conn
        except sqlite3.Error as e:
            logger.error(f"SQLite connection error: {e}")
            raise
        finally:
            conn.close()
            logger.info("SQLite connection closed")

    def execute_query(self, query: str) -> List[Dict[str, Any]]:
        """
        Execute a SELECT query and return results as a list of dictionaries.

        Args:
            query: SQL SELECT query string

        Returns:
            List of dictionaries representing rows
        """
        with self.get_connection() as conn:
            cursor = conn.cursor()
            try:
                cursor.execute(query)
                columns = [description[0] for description in cursor.description]
                rows = cursor.fetchall()
                results = [dict(row) for row in rows]
                logger.info(f"Query executed successfully. Returned {len(results)} rows")
                return results
            except Exception as e:
                logger.error(f"Query execution error: {e}")
                raise
            finally:
                cursor.close()

In [ ]:
class SQLValidator:
    """Validates SQL queries to ensure they are safe and read-only."""

    # Dangerous SQL keywords that should not appear in queries
    DANGEROUS_KEYWORDS = [
        'INSERT', 'UPDATE', 'DELETE', 'DROP', 'CREATE', 'ALTER',
        'TRUNCATE', 'EXEC', 'EXECUTE', 'GRANT', 'REVOKE', 'MERGE'
    ]

    def __init__(self):
        self.allowed_views = settings.allowed_views

    def validate(self, query: str) -> Tuple[bool, Optional[str]]:
        """
        Validate a SQL query for safety.

        Args:
            query: SQL query string to validate

        Returns:
            Tuple of (is_valid, error_message)
        """
        if not query or not query.strip():
            return False, "Query is empty"

        query_upper = query.upper().strip()

        # Must start with SELECT
        if not query_upper.startswith('SELECT'):
            return False, "Only SELECT queries are allowed"

        # Check for dangerous keywords
        for keyword in self.DANGEROUS_KEYWORDS:
            pattern = r'\b' + re.escape(keyword) + r'\b'
            if re.search(pattern, query_upper):
                return False, f"Dangerous keyword '{keyword}' is not allowed"

        # Check that only allowed views are referenced
        view_pattern = r'\bFROM\s+(\w+)\b'
        matches = re.findall(view_pattern, query_upper)

        if matches:
            referenced_views = [match.upper() for match in matches]
            allowed_upper = [view.upper() for view in self.allowed_views]

            for view in referenced_views:
                if view not in allowed_upper:
                    return False, f"View '{view}' is not in the allowed list"

        # Check for semicolons (potential SQL injection risk)
        if ';' in query:
            if query.rstrip().rstrip(';') != query.rstrip():
                return False, "Semicolons are only allowed at the end of queries"

        logger.info("SQL query validation passed")
        return True, None

    def sanitize(self, query: str) -> str:
        """Sanitize a SQL query by removing trailing semicolons and extra whitespace."""
        query = query.rstrip().rstrip(';')
        query = ' '.join(query.split())
        return query

In [ ]:
class SQLValidator:
    """Validates SQL queries to ensure they are safe and read-only."""

    # Dangerous SQL keywords that should not appear in queries
    DANGEROUS_KEYWORDS = [
        'INSERT', 'UPDATE', 'DELETE', 'DROP', 'CREATE', 'ALTER',
        'TRUNCATE', 'EXEC', 'EXECUTE', 'GRANT', 'REVOKE', 'MERGE'
    ]

    def __init__(self):
        self.allowed_views = settings.allowed_views

    def validate(self, query: str) -> Tuple[bool, Optional[str]]:
        """
        Validate a SQL query for safety.

        Args:
            query: SQL query string to validate

        Returns:
            Tuple of (is_valid, error_message)
        """
        if not query or not query.strip():
            return False, "Query is empty"

        query_upper = query.upper().strip()

        # Must start with SELECT
        if not query_upper.startswith('SELECT'):
            return False, "Only SELECT queries are allowed"

        # Check for dangerous keywords
        for keyword in self.DANGEROUS_KEYWORDS:
            pattern = r'\b' + re.escape(keyword) + r'\b'
            if re.search(pattern, query_upper):
                return False, f"Dangerous keyword '{keyword}' is not allowed"

        # Check that only allowed views are referenced
        view_pattern = r'\bFROM\s+(\w+)\b'
        matches = re.findall(view_pattern, query_upper)

        if matches:
            referenced_views = [match.upper() for match in matches]
            allowed_upper = [view.upper() for view in self.allowed_views]

            for view in referenced_views:
                if view not in allowed_upper:
                    return False, f"View '{view}' is not in the allowed list"

        # Note: Semicolons are removed during sanitization, so we don't need to check here
        # This is a safety check in case sanitization wasn't called
        if ';' in query:
            # Check if semicolon appears anywhere except at the very end
            query_stripped = query.rstrip()
            semicolon_pos = query_stripped.rfind(';')
            if semicolon_pos != -1 and semicolon_pos < len(query_stripped) - 1:
                # Semicolon is in the middle, not at the end
                return False, "Semicolons are only allowed at the end of queries"

        logger.info("SQL query validation passed")
        return True, None

    def sanitize(self, query: str) -> str:
        """Sanitize a SQL query by removing trailing semicolons and extra whitespace."""
        # Remove trailing semicolons
        query = query.rstrip().rstrip(';').strip()
        # Normalize whitespace (preserve structure but remove excessive spaces)
        # Replace multiple spaces with single space, but keep newlines for readability
        lines = query.split('\n')
        cleaned_lines = [' '.join(line.split()) for line in lines]
        query = ' '.join(cleaned_lines)
        return query

# ============================================
# CELL 7: Data Quality Explainer
# ============================================
class DataQualityExplainer:
    """Generates rule-based explanations for data profiling results."""

    NULL_PERCENTAGE_HIGH = 50.0
    NULL_PERCENTAGE_MODERATE = 10.0
    DISTINCT_COUNT_LOW = 2

    def explain(self, results: List[Dict[str, Any]], question: str) -> str:
        """Generate rule-based explanation for query results."""
        if not results:
            return "No data found matching your query."

        explanations = []
        for row in results:
            row_explanation = self._explain_row(row)
            if row_explanation:
                explanations.append(row_explanation)

        if explanations:
            return " | ".join(explanations)

        return f"Found {len(results)} result(s). Data appears to be within normal parameters."

    def _explain_row(self, row: Dict[str, Any]) -> Optional[str]:
        """Generate explanation for a single row of results."""
        explanations = []

        null_pct = self._get_numeric_value(row, 'null_percentage')
        if null_pct is not None:
            if null_pct > self.NULL_PERCENTAGE_HIGH:
                explanations.append("High null percentage indicates data quality issue")
            elif null_pct > self.NULL_PERCENTAGE_MODERATE:
                explanations.append("Moderate null percentage requires data cleaning")
            elif null_pct > 0:
                explanations.append("Low null percentage - good data quality")
            else:
                explanations.append("No null values - excellent data quality")

        distinct_count = self._get_numeric_value(row, 'distinct_count')
        if distinct_count is not None and distinct_count < self.DISTINCT_COUNT_LOW:
            explanations.append("Low distinct count suggests limited data diversity")

        table_name = row.get('table_name', '')
        column_name = row.get('column_name', '')

        if table_name and column_name:
            context = f"{table_name}.{column_name}"
        elif table_name:
            context = table_name
        elif column_name:
            context = column_name
        else:
            context = None

        if explanations and context:
            return f"{context}: {', '.join(explanations)}"
        elif explanations:
            return ', '.join(explanations)

        return None

    def _get_numeric_value(self, row: Dict[str, Any], key: str) -> Optional[float]:
        """Safely extract numeric value from row."""
        value = row.get(key)
        if value is None:
            return None
        try:
            return float(value)
        except (ValueError, TypeError):
            return None

In [ ]:
class QueryRouter:
    """Routes queries to appropriate handlers (SQL vs General NLP)."""

    def route(self, question: str) -> Tuple[str, bool]:
        """
        Route a question to determine if it needs SQL generation.

        Returns:
            Tuple of (query_type, needs_sql)
        """
        question_lower = question.lower().strip()

        # Profiling-related keywords that should trigger SQL queries
        profiling_keywords = [
            'column', 'columns', 'table', 'tables', 'row', 'rows',
            'null', 'percentage', 'percentage', 'distinct', 'count',
            'profiling', 'statistics', 'data quality', 'data type', 'data types',
            'null values', 'null percentage', 'high null', 'low distinct',
            'table size', 'row count', 'column count'
        ]

        # Check if question contains profiling-related keywords
        has_profiling_keywords = any(keyword in question_lower for keyword in profiling_keywords)

        # Specific data request patterns
        is_data_request = any([
            'which columns' in question_lower,
            'which tables' in question_lower,
            'show me' in question_lower and ('table' in question_lower or 'column' in question_lower),
            'what are the' in question_lower and ('columns' in question_lower or 'tables' in question_lower or 'data types' in question_lower),
            'list' in question_lower and ('columns' in question_lower or 'tables' in question_lower),
            'find' in question_lower and ('columns' in question_lower or 'tables' in question_lower),
            'get' in question_lower and ('table' in question_lower or 'column' in question_lower),
            'query' in question_lower and ('table' in question_lower or 'column' in question_lower),
        ])

        # Explicit SQL patterns
        explicit_sql_patterns = [
            r'\b(show|list|find|get|query|select).*\b(table|column|row|data).*\b(null|percentage|count|distinct|type)',
            r'\bwhich\s+(columns?|tables?|rows?).*\b(have|with|contain|show)',
            r'\bwhat\s+(are|is)\s+the\s+(columns?|tables?|rows?|data\s+types?)\s+(in|of|with)',
        ]
        has_explicit_sql = any(re.search(pattern, question_lower) for pattern in explicit_sql_patterns)

        # General conversation indicators (very limited - only for clear conversational queries)
        general_indicators = [
            question_lower in ['hello', 'hi', 'hey', 'help'],
            question_lower.startswith('what can you do'),
            len(question.split()) <= 2 and question_lower.endswith('?'),
        ]

        has_general = any(general_indicators)

        # Route to SQL if it's a data request, has explicit SQL patterns, or contains profiling keywords
        # Only route to general NLP for very clear conversational queries
        if has_general and not has_profiling_keywords:
            return ('general', False)
        elif is_data_request or has_explicit_sql or has_profiling_keywords:
            return ('sql', True)
        else:
            # Default to general for ambiguous queries
            return ('general', False)

In [ ]:
class LLMClient:
    """Client for interacting with Phi-3 model via transformers."""

    def __init__(self, pipeline_instance):
        self.pipe = pipeline_instance

    def _generate_text(self, prompt: str, max_new_tokens: int = 200, temperature: float = 0.3) -> str:
        """Generate text using the pipeline."""
        messages = [{"role": "user", "content": prompt}]
        try:
            out = self.pipe(messages, max_new_tokens=max_new_tokens, temperature=temperature, do_sample=True)
            generated_text = out[0]["generated_text"]

            # Extract the assistant's response - handle different formats
            if isinstance(generated_text, list):
                # If it's a list of dicts, find the assistant's message
                for item in reversed(generated_text):
                    if isinstance(item, dict) and item.get("role") == "assistant":
                        return item.get("content", "").strip()
                # If no assistant found, return the last item's content
                if generated_text and isinstance(generated_text[-1], dict):
                    return generated_text[-1].get("content", "").strip()

            elif isinstance(generated_text, str):
                # Try to parse if it's a string representation of a list/dict
                if generated_text.strip().startswith('[') and "'role'" in generated_text:
                    try:
                        import ast
                        parsed = ast.literal_eval(generated_text)
                        if isinstance(parsed, list):
                            for item in reversed(parsed):
                                if isinstance(item, dict) and item.get("role") == "assistant":
                                    return item.get("content", "").strip()
                    except:
                        pass  # Fall through to other extraction methods

                # Try to extract using assistant marker
                if "<|assistant|>" in generated_text:
                    parts = generated_text.split("<|assistant|>")
                    if len(parts) > 1:
                        response = parts[-1].strip()
                        response = response.replace("<|end|>", "").strip()
                        if response:
                            return response

                # Try to find where the prompt ends and response begins
                if prompt in generated_text:
                    prompt_pos = generated_text.find(prompt)
                    if prompt_pos != -1:
                        response_start = prompt_pos + len(prompt)
                        response_text = generated_text[response_start:].strip()
                        # Remove common formatting markers
                        for marker in ["<|end|>", "<|assistant|>", "\n\n", "<|user|>"]:
                            if response_text.startswith(marker):
                                response_text = response_text[len(marker):].strip()
                        if response_text:
                            return response_text

                # Return as-is if extraction failed
                return generated_text.strip()

            # Fallback: convert to string
            return str(generated_text).strip()
        except Exception as e:
            logger.error(f"LLM generation error: {e}")
            return ""

    def generate_sql(self, user_question: str) -> Optional[str]:
        """Generate SQL query from natural language question."""
        allowed_views = ", ".join(settings.allowed_views)

        prompt = f"""You are a SQL query generator for data profiling statistics.

Available views:
- {allowed_views}

These views contain data profiling information with columns like:
- table_name
- column_name
- null_percentage
- distinct_count
- data_type
- row_count

Generate a SQL SELECT query to answer this question: {user_question}

Rules:
1. Only use SELECT statements
2. Only query from the allowed views listed above
3. Return only the SQL query, no explanations
4. Use SQLite syntax (LIMIT instead of TOP, no semicolons needed)

SQL Query:"""

        try:
            response = self._generate_text(prompt, max_new_tokens=150, temperature=0.1)
            sql_query = self._extract_sql_from_response(response)
            logger.info(f"Generated SQL query: {sql_query}")
            return sql_query
        except Exception as e:
            logger.error(f"SQL generation error: {e}")
            return None

    def summarize_results(self, question: str, results: list, explanation: str) -> str:
        """Generate a natural language summary of query results."""
        results_str = str(results[:10])  # Limit to first 10 rows

        prompt = f"""User asked: {question}

Query returned {len(results)} results.

Data quality assessment: {explanation}

Provide a concise, natural language answer (2-3 sentences) explaining what the data shows:"""

        try:
            summary = self._generate_text(prompt, max_new_tokens=150, temperature=0.3)
            logger.info("Generated summary from LLM")
            return summary
        except Exception as e:
            logger.error(f"Summarization error: {e}")
            return explanation

    def process_general_query(self, question: str, context: Optional[dict] = None) -> str:
        """Process general NLP queries."""
        base_prompt = f"""You are a friendly and knowledgeable data profiling assistant. You help users understand data quality, analytics, and answer questions in a natural, conversational way.

User question: {question}
"""

        if context:
            context_info = "\nContext:\n"
            if context.get('available_tables'):
                context_info += f"Available tables: {', '.join(context['available_tables'])}\n"
            if context.get('database_info'):
                context_info += f"Database: {context['database_info']}\n"
            base_prompt += context_info

        base_prompt += """
Instructions:
- Be conversational, friendly, and helpful
- Answer naturally as if you're having a conversation
- If asked about data profiling concepts, explain them clearly with examples
- If the question is about specific data in the database, you can offer to help query it
- Feel free to ask clarifying questions if needed
- Keep responses concise but informative
- You can discuss data quality, analytics, best practices, and general data topics

Provide a natural, helpful response:"""

        try:
            answer = self._generate_text(base_prompt, max_new_tokens=200, temperature=0.8)
            logger.info("Generated general NLP response")
            return answer
        except Exception as e:
            logger.error(f"General NLP error: {e}")
            return "I apologize, but I'm having trouble processing your request right now. Please try again."

    def _extract_sql_from_response(self, response: str) -> str:
        """Extract SQL query from LLM response (handles markdown code blocks)."""
        sql_pattern = r'```(?:sql)?\s*(.*?)```'
        match = re.search(sql_pattern, response, re.DOTALL | re.IGNORECASE)

        if match:
            return match.group(1).strip()

        if 'SELECT' in response.upper():
            start = response.upper().find('SELECT')
            sql = response[start:]
            lines = sql.split('\n')
            sql_lines = []
            for line in lines:
                if line.strip() and not line.strip().startswith('```'):
                    sql_lines.append(line)
                if line.strip().endswith(';'):
                    break
            return ' '.join(sql_lines).rstrip(';').strip()

        return response.strip()

In [ ]:
def create_sample_database(db_path: str = "profiling_sample.db"):
    """Create SQLite database with sample profiling data."""
    import os
    if os.path.exists(db_path):
        os.remove(db_path)
        print(f"Removed existing database: {db_path}")

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    print("Creating database schema...")

    cursor.execute("""
        CREATE TABLE profiling_tables (
            table_id INTEGER PRIMARY KEY AUTOINCREMENT,
            table_name TEXT NOT NULL,
            row_count INTEGER,
            column_count INTEGER,
            last_profiled_date TIMESTAMP,
            table_size_mb REAL
        )
    """)

    cursor.execute("""
        CREATE TABLE profiling_columns (
            column_id INTEGER PRIMARY KEY AUTOINCREMENT,
            table_id INTEGER,
            column_name TEXT NOT NULL,
            data_type TEXT,
            null_count INTEGER,
            distinct_count INTEGER,
            avg_length REAL,
            min_value TEXT,
            max_value TEXT,
            duplicate_count INTEGER,
            FOREIGN KEY (table_id) REFERENCES profiling_tables(table_id)
        )
    """)

    print("Inserting sample table data...")
    tables_data = [
        ("customers", 15000, 12, datetime.now() - timedelta(days=1), 2.5),
        ("orders", 45000, 8, datetime.now() - timedelta(days=2), 3.8),
        ("products", 2500, 15, datetime.now() - timedelta(days=1), 1.2),
        ("transactions", 125000, 10, datetime.now() - timedelta(hours=12), 8.5),
        ("employees", 500, 20, datetime.now() - timedelta(days=3), 0.8),
    ]

    for table_name, row_count, col_count, last_profiled, size_mb in tables_data:
        cursor.execute("""
            INSERT INTO profiling_tables (table_name, row_count, column_count, last_profiled_date, table_size_mb)
            VALUES (?, ?, ?, ?, ?)
        """, (table_name, row_count, col_count, last_profiled, size_mb))

    print("Inserting sample column profiling data...")

    customers_columns = [
        (1, "customer_id", "INTEGER", 0, 15000, None, "1", "15000", 0),
        (1, "first_name", "VARCHAR", 0, 14200, 12.5, "Aaron", "Zoe", 800),
        (1, "last_name", "VARCHAR", 0, 14800, 15.2, "Anderson", "Zimmerman", 200),
        (1, "email", "VARCHAR", 1500, 13500, 24.8, "a@example.com", "z@example.com", 0),
        (1, "phone", "VARCHAR", 3200, 12000, 13.0, "100-000-0000", "999-999-9999", 3000),
        (1, "address", "VARCHAR", 800, 14500, 45.2, "100 Main St", "9999 Oak Ave", 500),
        (1, "city", "VARCHAR", 0, 850, 12.5, "Albany", "Zurich", 0),
        (1, "state", "VARCHAR", 0, 52, 2.0, "AK", "WY", 0),
        (1, "zip_code", "VARCHAR", 0, 1200, 5.0, "00001", "99999", 0),
        (1, "registration_date", "DATE", 0, 1825, None, "2019-01-01", "2024-12-31", 0),
        (1, "status", "VARCHAR", 0, 3, 6.0, "active", "inactive", 0),
        (1, "notes", "TEXT", 8500, 8500, 125.5, None, None, 0),
    ]

    orders_columns = [
        (2, "order_id", "INTEGER", 0, 45000, None, "1", "45000", 0),
        (2, "customer_id", "INTEGER", 0, 12000, None, "1", "15000", 0),
        (2, "order_date", "DATE", 0, 1095, None, "2022-01-01", "2024-12-31", 0),
        (2, "total_amount", "DECIMAL", 0, 12500, None, "5.99", "9999.99", 0),
        (2, "status", "VARCHAR", 0, 5, 10.0, "pending", "shipped", 0),
        (2, "shipping_address", "VARCHAR", 500, 44000, 48.5, None, None, 1000),
        (2, "payment_method", "VARCHAR", 0, 8, 12.0, "cash", "wire_transfer", 0),
        (2, "notes", "TEXT", 35000, 10000, 85.2, None, None, 0),
    ]

    products_columns = [
        (3, "product_id", "INTEGER", 0, 2500, None, "1", "2500", 0),
        (3, "product_name", "VARCHAR", 0, 2500, 28.5, "Widget A", "Zebra Stripes", 0),
        (3, "category", "VARCHAR", 0, 25, 15.0, "Electronics", "Toys", 0),
        (3, "price", "DECIMAL", 0, 1250, None, "0.99", "999.99", 0),
        (3, "stock_quantity", "INTEGER", 0, 500, None, "0", "10000", 0),
        (3, "description", "TEXT", 200, 2300, 145.8, None, None, 0),
        (3, "supplier_id", "INTEGER", 500, 200, None, "1", "50", 0),
        (3, "created_date", "DATE", 0, 1825, None, "2019-01-01", "2024-12-31", 0),
        (3, "is_active", "BOOLEAN", 0, 2, None, "0", "1", 0),
        (3, "tags", "VARCHAR", 800, 1700, 25.5, None, None, 0),
        (3, "image_url", "VARCHAR", 1200, 1300, 45.2, None, None, 0),
        (3, "weight_kg", "DECIMAL", 300, 2200, None, "0.01", "50.00", 0),
        (3, "dimensions", "VARCHAR", 400, 2100, 18.5, None, None, 0),
        (3, "warranty_months", "INTEGER", 600, 25, None, "0", "60", 0),
        (3, "rating", "DECIMAL", 1500, 100, None, "1.0", "5.0", 0),
    ]

    transactions_columns = [
        (4, "transaction_id", "INTEGER", 0, 125000, None, "1", "125000", 0),
        (4, "order_id", "INTEGER", 0, 45000, None, "1", "45000", 0),
        (4, "transaction_date", "TIMESTAMP", 0, 87500, None, "2022-01-01 00:00:00", "2024-12-31 23:59:59", 0),
        (4, "amount", "DECIMAL", 0, 8750, None, "0.01", "5000.00", 0),
        (4, "currency", "VARCHAR", 0, 5, 3.0, "EUR", "USD", 0),
        (4, "payment_status", "VARCHAR", 0, 4, 10.0, "failed", "success", 0),
        (4, "processor_response", "TEXT", 25000, 100000, 125.5, None, None, 0),
        (4, "refund_amount", "DECIMAL", 110000, 1500, None, "0.00", "5000.00", 0),
        (4, "fraud_score", "DECIMAL", 50000, 75000, None, "0.0", "100.0", 0),
        (4, "metadata", "JSON", 30000, 95000, 85.2, None, None, 0),
    ]

    employees_columns = [
        (5, "employee_id", "INTEGER", 0, 500, None, "1", "500", 0),
        (5, "first_name", "VARCHAR", 0, 485, 8.5, "Alice", "Zachary", 0),
        (5, "last_name", "VARCHAR", 0, 495, 10.2, "Adams", "Zimmer", 0),
        (5, "email", "VARCHAR", 0, 500, 22.5, "a.adams@company.com", "z.zimmer@company.com", 0),
        (5, "department", "VARCHAR", 0, 12, 15.0, "Engineering", "Sales", 0),
        (5, "position", "VARCHAR", 0, 45, 20.5, "Intern", "VP Engineering", 0),
        (5, "salary", "DECIMAL", 0, 450, None, "40000.00", "250000.00", 0),
        (5, "hire_date", "DATE", 0, 1825, None, "2019-01-01", "2024-12-31", 0),
        (5, "manager_id", "INTEGER", 50, 450, None, "1", "500", 0),
        (5, "phone_extension", "VARCHAR", 100, 400, 4.0, "1000", "9999", 0),
        (5, "office_location", "VARCHAR", 0, 8, 12.0, "Building A", "Remote", 0),
        (5, "emergency_contact", "VARCHAR", 200, 300, 35.5, None, None, 0),
        (5, "emergency_phone", "VARCHAR", 200, 300, 13.0, None, None, 0),
        (5, "start_date", "DATE", 0, 1825, None, "2019-01-01", "2024-12-31", 0),
        (5, "end_date", "DATE", 450, 50, None, None, None, 0),
        (5, "status", "VARCHAR", 0, 3, 8.0, "active", "terminated", 0),
        (5, "notes", "TEXT", 400, 100, 125.5, None, None, 0),
        (5, "performance_rating", "DECIMAL", 150, 350, None, "1.0", "5.0", 0),
        (5, "training_completed", "BOOLEAN", 0, 2, None, "0", "1", 0),
        (5, "certifications", "VARCHAR", 300, 200, 45.2, None, None, 0),
    ]

    all_columns = customers_columns + orders_columns + products_columns + transactions_columns + employees_columns

    cursor.executemany("""
        INSERT INTO profiling_columns
        (table_id, column_name, data_type, null_count, distinct_count, avg_length, min_value, max_value, duplicate_count)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, all_columns)

    print("Creating profiling views...")

    cursor.execute("""
        CREATE VIEW profiling_column_stats AS
        SELECT
            t.table_name,
            c.column_name,
            CAST(c.null_count * 100.0 / NULLIF(t.row_count, 0) AS REAL) AS null_percentage,
            c.distinct_count,
            c.data_type,
            t.row_count,
            c.avg_length,
            c.min_value,
            c.max_value
        FROM profiling_tables t
        INNER JOIN profiling_columns c ON t.table_id = c.table_id
    """)

    cursor.execute("""
        CREATE VIEW profiling_table_stats AS
        SELECT
            table_name,
            row_count,
            column_count,
            last_profiled_date AS last_updated,
            table_size_mb
        FROM profiling_tables
    """)

    cursor.execute("""
        CREATE VIEW profiling_data_quality AS
        SELECT
            t.table_name,
            c.column_name,
            CAST(c.null_count * 100.0 / NULLIF(t.row_count, 0) AS REAL) AS null_percentage,
            CAST(c.duplicate_count * 100.0 / NULLIF(t.row_count, 0) AS REAL) AS duplicate_percentage,
            CASE
                WHEN c.null_count * 100.0 / NULLIF(t.row_count, 0) > 50 THEN 'Poor'
                WHEN c.null_count * 100.0 / NULLIF(t.row_count, 0) > 10 THEN 'Fair'
                ELSE 'Good'
            END AS quality_score
        FROM profiling_tables t
        INNER JOIN profiling_columns c ON t.table_id = c.table_id
    """)

    conn.commit()
    conn.close()

    print(f"\n✅ Database created successfully: {db_path}")
    print(f"✅ Inserted {len(tables_data)} tables")
    print(f"✅ Inserted {len(all_columns)} columns")
    print(f"✅ Created 3 profiling views")
    print("\nDatabase is ready for use!")

# Initialize database
create_sample_database("profiling_sample.db")


Creating database schema...
Inserting sample table data...
Inserting sample column profiling data...
Creating profiling views...

✅ Database created successfully: profiling_sample.db
✅ Inserted 5 tables
✅ Inserted 65 columns
✅ Created 3 profiling views

Database is ready for use!


/tmp/ipython-input-479727819.py:50: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("""


In [ ]:
db = DatabaseConnection("profiling_sample.db")
sql_validator = SQLValidator()
explainer = DataQualityExplainer()
query_router = QueryRouter()
llm_client = LLMClient(pipe)

print("✅ All components initialized!")

✅ All components initialized!


In [ ]:
def chat(question: str) -> Dict[str, Any]:
    """
    Main chat function that handles both SQL queries and general NLP questions.

    Args:
        question: User's question

    Returns:
        Dictionary with answer, sql_query (if applicable), results_count, and explanation
    """
    question = question.strip()

    if not question:
        return {"answer": "Question cannot be empty", "error": True}

    logger.info(f"Received question: {question}")

    # Route the query
    query_type, needs_sql = query_router.route(question)
    logger.info(f"Query routed as: {query_type} (needs_sql: {needs_sql})")

    try:
        if needs_sql:
            # SQL Query Path
            return _handle_sql_query(question)
        else:
            # General NLP Path
            return _handle_general_query(question)
    except Exception as e:
        logger.error(f"Unexpected error: {e}", exc_info=True)
        return {
            "answer": f"Error: {str(e)}",
            "error": True
        }

def _handle_sql_query(question: str) -> Dict[str, Any]:
    """Handle SQL query generation and execution."""
    # Step 1: Generate SQL from natural language
    sql_query = llm_client.generate_sql(question)

    if not sql_query:
        logger.error(f"SQL generation returned empty for question: {question}")
        return {
            "answer": "Failed to generate SQL query. Please try rephrasing your question or be more specific about what profiling data you want to see.",
            "error": True
        }

    logger.info(f"Generated SQL query (before sanitization): {sql_query}")

    # Sanitize query first (remove semicolons, normalize whitespace)
    sql_query = sql_validator.sanitize(sql_query)
    logger.info(f"Generated SQL query (after sanitization): {sql_query}")

    # Step 2: Validate SQL query
    is_valid, error_message = sql_validator.validate(sql_query)

    if not is_valid:
        logger.warning(f"SQL validation failed: {error_message}. Query: {sql_query}")
        return {
            "answer": f"Generated SQL query is not safe: {error_message}. Please try rephrasing your question.",
            "error": True
        }

    # Step 3: Execute query
    try:
        results = db.execute_query(sql_query)
        logger.info(f"Query executed successfully. Returned {len(results)} rows")
    except Exception as e:
        logger.error(f"Query execution failed: {e}. SQL: {sql_query}")
        return {
            "answer": f"Database query failed: {str(e)}. The generated SQL query was: {sql_query}",
            "error": True,
            "sql_query": sql_query  # Include SQL query in error for debugging
        }

    # Step 4: Generate rule-based explanation
    explanation = explainer.explain(results, question)

    # Step 5: Generate natural language summary
    answer = llm_client.summarize_results(question, results, explanation)

    # Fallback to explanation if LLM summarization fails
    if not answer or len(answer.strip()) < 10:
        answer = explanation

    logger.info(f"Successfully processed SQL question. Returned {len(results)} results")

    return {
        "answer": answer,
        "sql_query": sql_query,
        "results_count": len(results),
        "explanation": explanation,
        "error": False
    }

def _handle_general_query(question: str) -> Dict[str, Any]:
    """Handle general NLP queries with conversational responses."""
    # Get context about available data (optional, non-blocking)
    context = {}
    try:
        tables_result = db.execute_query(
            "SELECT DISTINCT table_name FROM profiling_table_stats LIMIT 10"
        )
        context['available_tables'] = [row['table_name'] for row in tables_result]
        context['database_info'] = "Sample profiling database with data quality metrics"
    except:
        pass

    # Process with general NLP
    answer = llm_client.process_general_query(question, context)

    logger.info("Successfully processed general NLP question")

    return {
        "answer": answer,
        "sql_query": None,
        "results_count": 0,
        "explanation": None,
        "error": False
    }

In [ ]:
try:
    import gradio as gr
except ImportError:
    print("⚠️ Gradio not installed. Installing now...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio"])
    import gradio as gr

def format_response(response: Dict[str, Any]) -> str:
    """Format the chat response for display in the UI."""
    if response.get("error"):
        return f"❌ **Error:** {response['answer']}"

    formatted = response['answer']

    # Add SQL query if present
    if response.get("sql_query"):
        formatted += f"\n\n**📝 SQL Query:**\n```sql\n{response['sql_query']}\n```"

    # Add metadata if present
    metadata_parts = []
    if response.get("results_count", 0) > 0:
        metadata_parts.append(f"**Results:** {response['results_count']} rows")
    if response.get("explanation"):
        metadata_parts.append(f"**Explanation:** {response['explanation']}")

    if metadata_parts:
        formatted += f"\n\n💡 {' | '.join(metadata_parts)}"

    return formatted

def chat_interface(message, history):
    """Gradio chat interface handler.

    Args:
        message: Current user message
        history: List of [user_message, assistant_message] pairs

    Returns:
        Assistant's response string
    """
    if not message or not message.strip():
        return ""

    # Process the message using the chat function
    response = chat(message)
    formatted_response = format_response(response)

    return formatted_response

def interactive_chat():
    """Launch Gradio chat interface."""
    print("🚀 Launching Data Profiling Chatbot UI...")
    print("📊 Ask questions about your data profiling statistics")
    print("\n" + "="*60 + "\n")

    # Create Gradio ChatInterface
    demo = gr.ChatInterface(
        fn=chat_interface,
        title="📊 Data Profiling Chatbot",
        description="Ask questions about your data profiling statistics. I can help you query database statistics, explain data quality metrics, and answer general questions about data profiling.",
        examples=[
            "Which columns have high null values?",
            "Show me tables with the most rows",
            "What are the data types in the customer table?",
            "Which columns have low distinct counts?",
            "Hello! What can you do?",
            "Explain data profiling"
        ]
    )

    # Launch in Colab
    demo.launch(share=True, height=600)

    print("\n✅ UI launched! Interact with the chat interface above.\n")

In [ ]:
question = "Which columns have high null values?"
print(f"Question: {question}\n")
response = chat(question)
print(f"Answer: {response['answer']}\n")
if response.get("sql_query"):
    print(f"SQL Query: {response['sql_query']}\n")
if response.get("results_count", 0) > 0:
    print(f"Results: {response['results_count']} rows\n")


Question: Which columns have high null values?

Answer: The data indicates that several columns, including'refund_amount', 'end_date', and 'certifications', have a high percentage of null values, suggesting potential data quality issues that may need to be addressed.

SQL Query: SELECT column_name, null_percentage  FROM profiling_column_stats  WHERE null_percentage > 50  LIMIT 10

Results: 7 rows



In [ ]:
interactive_chat()

🚀 Launching Data Profiling Chatbot UI...
📊 Ask questions about your data profiling statistics


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4a5aa6c21d575ea810.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



✅ UI launched! Interact with the chat interface above.

